# Modul Pembelajaran Adversarial Search RKA 25

## 1. Pendahuluan</b></h2>

Pada modul sebelumnya, kita telah mempelajari **Uninformed Search** (BFS, DFS, UCS) dan **Informed Search** (GBFS, A\*). Algoritma-algoritma tersebut bekerja dengan asumsi bahwa agen bergerak sendiri tanpa ada lawan yang mempengaruhi keputusan.

**Adversarial Search** hadir untuk menjawab pertanyaan:
> *Bagaimana jika ada lawan yang juga bermain secara cerdas dan selalu berusaha mengalahkan kita?*

Algoritma Adversarial Search dirancang untuk situasi **kompetitif (multi-agent)**, di mana setiap agen memiliki tujuan yang **berlawanan** satu sama lain.

Dua algoritma utama yang akan dipelajari:
1. **Minimax** — algoritma dasar adversarial search
2. **Alpha-Beta Pruning** — optimasi dari Minimax dengan memangkas cabang yang tidak perlu

## 2. Konsep Dasar

### 2.1 Zero-Sum Game

Adversarial Search paling sering diterapkan pada **Zero-Sum Game**, yaitu permainan di mana:
- Keuntungan satu pemain = Kerugian pemain lain
- Tidak ada hasil yang menguntungkan kedua pihak secara bersamaan (*no win-win outcome*)

Contoh nyata Zero-Sum Game:
- **Catur** — Deep Blue (IBM, 1997) mengalahkan juara dunia Garry Kasparov
- **Checkers** — Chinook (1994) menjadi juara dunia komputer pertama
- **Go** — AlphaGo (Google DeepMind, 2016) mengalahkan juara dunia manusia
- **Tic-Tac-Toe** — contoh paling sederhana untuk dipelajari

### 2.2 Elemen Formal sebuah Game

Sebuah game dalam AI secara formal didefinisikan dengan komponen berikut:

| Elemen | Deskripsi |
|---|---|
| **S₀** | State awal (Initial State) |
| **TO-MOVE(s)** | Pemain yang mendapat giliran pada state *s* |
| **ACTIONS(s)** | Sekumpulan aksi yang valid pada state *s* |
| **RESULT(s, a)** | State baru hasil dari aksi *a* di state *s* |
| **IS-TERMINAL(s)** | Bernilai True jika game selesai, False jika belum |
| **UTILITY(s, p)** | Nilai numerik untuk pemain *p* saat game berakhir di state *s* |

Pada catur misalnya, **UTILITY** bernilai: **+1** (menang), **-1** (kalah), **0** (seri).

### 2.3 Game Tree

**Game Tree** adalah representasi semua kemungkinan langkah dalam sebuah permainan.

- **Root** = State saat ini
- **Branch** = Setiap aksi yang mungkin dilakukan
- **Leaf Node** = Terminal state (akhir permainan)
- **Layer bergantian** antara giliran MAX dan MIN

Konsep layer dalam game tree:
- **Layer MAX (Maximizer)** = Giliran komputer/agen kita → memilih nilai **tertinggi**
- **Layer MIN (Minimizer)** = Giliran lawan → memilih nilai **terendah**

```
         [MAX]          ← Root: giliran kita
        /     \
     [MIN]   [MIN]      ← Giliran lawan
     /  \    /  \
   [3]  [5] [2]  [9]   ← Terminal state (leaf nodes)
```

## 3. Algoritma Minimax
![image.png](attachment:image.png)

### 3.1 Penjelasan Minimax

**Minimax** adalah algoritma rekursif yang bekerja dengan cara:
1. Membangun seluruh game tree hingga terminal state
2. Mengevaluasi nilai setiap terminal state menggunakan **utility function**
3. Mem-*backup* nilai ke atas tree secara bergantian:
   - **MAX node**: ambil nilai **maksimum** dari semua anak
   - **MIN node**: ambil nilai **minimum** dari semua anak
4. Keputusan akhir = aksi dari root yang mengarah ke nilai tertinggi

Formula:
```
MINIMAX(s) = UTILITY(s)                        jika IS-TERMINAL(s)
           = max(MINIMAX(RESULT(s,a)))          jika giliran MAX
           = min(MINIMAX(RESULT(s,a)))          jika giliran MIN
```

**Properti Minimax:**
- **Complete**: Ya, jika tree berhingga
- **Optimal**: Ya, asalkan lawan juga bermain secara optimal
- **Time Complexity**: O(b^m) — b = branching factor, m = kedalaman tree
- **Space Complexity**: O(bm)

### 3.2 Contoh Manual Minimax

Perhatikan game tree berikut:

```
            [?]          ← MAX (root)
           / | \
        [?] [?] [?]    ← MIN
        /\   |   /\
       3  5  2  9  1    ← Terminal values
```

**Langkah penyelesaian:**

1. MIN node kiri: min(3, 5) = **3**
2. MIN node tengah: min(2) = **2**  
3. MIN node kanan: min(9, 1) = **1**
4. MAX node (root): max(3, 2, 1) = **3** → pilih cabang kiri

Komputer akan memilih **cabang kiri** karena menghasilkan nilai terbaik (3) bahkan dengan asumsi lawan bermain optimal.

### 3.3 Representasi Game Tree

Untuk memahami Minimax, kita gunakan representasi sederhana berupa **game tree**.

* Node = state
* Edge = pilihan langkah
* Leaf node = nilai akhir (utility)

Pada implementasi ini, tree direpresentasikan sebagai dictionary:

* Key = node
* Value = list child atau nilai (jika leaf)


In [1]:
tree = {
    'A': ['B', 'C', 'D'],

    'B': ['E', 'F'],
    'C': ['G', 'H'],
    'D': ['I', 'J'],

    'E': [8, 5],
    'F': [6, -4],
    'G': [3, 8],
    'H': [4, -6],
    'I': [1, 2],
    'J': [5, 2]
}

### 3.4 Implementasi Minimax

Berikut implementasi Minimax berdasarkan pseudocode:

* Jika mencapai leaf → kembalikan nilai
* Jika MAX → ambil nilai maksimum dari child
* Jika MIN → ambil nilai minimum dari child


In [2]:
def minimax(node, depth, maximizingPlayer):

    # Base case
    if node not in tree or depth == 0:
        return node

    if maximizingPlayer:
        bestValue = float('-inf')
        for child in tree[node]:
            value = minimax(child, depth - 1, False)
            bestValue = max(bestValue, value)
        return bestValue

    else:
        bestValue = float('inf')
        for child in tree[node]:
            value = minimax(child, depth - 1, True)
            bestValue = min(bestValue, value)
        return bestValue

In [3]:
def find_best_move(node, depth):
    bestValue = float('-inf')
    bestMove = None

    for child in tree[node]:
        value = minimax(child, depth - 1, False)
        if value > bestValue:
            bestValue = value
            bestMove = child
    return bestMove, bestValue

In [4]:
move, value = find_best_move('A', 3)

print("Best move:", move)
print("Best value:", value)

Best move: B
Best value: 6


### 3.4 Visualisasi Proses Minimax

Agar lebih mudah dipahami, kita tampilkan proses traversal Minimax.

* kita bisa melihat bagaimana nilai dihitung
* memahami keputusan MAX dan MIN


In [5]:
def minimax_trace(node, depth, maximizingPlayer, indent=""):

    if node not in tree or depth == 0:
        print(f"{indent}{node}")
        return node
    print(f"{indent}{node} ({'MAX' if maximizingPlayer else 'MIN'})")

    if maximizingPlayer:
        bestValue = float('-inf')
        for child in tree[node]:
            value = minimax_trace(child, depth - 1, False, indent + "  ")
            bestValue = max(bestValue, value)
        print(f"{indent}-> pilih {bestValue}")
        return bestValue

    else:
        bestValue = float('inf')
        for child in tree[node]:
            value = minimax_trace(child, depth - 1, True, indent + "  ")
            bestValue = min(bestValue, value)

        print(f"{indent}-> pilih {bestValue}")
        return bestValue

In [6]:
result = minimax_trace('A', depth=3, maximizingPlayer=True)
print("\nNilai optimal:", result)

A (MAX)
  B (MIN)
    E (MAX)
      8
      5
    -> pilih 8
    F (MAX)
      6
      -4
    -> pilih 6
  -> pilih 6
  C (MIN)
    G (MAX)
      3
      8
    -> pilih 8
    H (MAX)
      4
      -6
    -> pilih 4
  -> pilih 4
  D (MIN)
    I (MAX)
      1
      2
    -> pilih 2
    J (MAX)
      5
      2
    -> pilih 5
  -> pilih 2
-> pilih 6

Nilai optimal: 6


## 4. Alpha-Beta Pruning

![Alpha_Beta_gif.gif](attachment:Alpha_Beta_gif.gif)

### 4.1 Mengapa Perlu Pruning?

Minimax tetap menjamin keputusan optimal, tetapi ia tetap harus mengevaluasi hampir seluruh game tree. Pada tree kecil, hal ini masih mudah; pada tree yang lebih besar, jumlah node yang dicek bisa tumbuh sangat cepat.

**Alpha-Beta Pruning** adalah optimasi Minimax untuk menghentikan eksplorasi cabang yang sudah pasti tidak akan memengaruhi hasil akhir.

Intinya: jika sebuah cabang sudah lebih buruk daripada batas terbaik yang sudah ditemukan, cabang itu tidak perlu dihitung sampai selesai.

### 4.2 Penjelasan Alpha-Beta Pruning

Alpha-Beta Pruning menambahkan dua parameter ke dalam Minimax:

| Parameter | Makna | Digunakan oleh |
|---|---|---|
| **Alpha (α)** | Nilai terbaik yang sudah ditemukan MAX di jalur saat ini | MAX |
| **Beta (β)** | Nilai terbaik yang sudah ditemukan MIN di jalur saat ini | MIN |

**Kondisi Pruning:**
- Pada node **MAX**, jika nilai sementara sudah ≥ β, maka sisa anak tidak perlu dievaluasi lagi karena MIN sudah punya pilihan yang lebih baik atau sama baiknya di jalur lain.
- Pada node **MIN**, jika nilai sementara sudah ≤ α, maka sisa anak tidak perlu dievaluasi lagi karena MAX sudah punya pilihan yang lebih baik atau sama baiknya di jalur lain.

Intuisi: *alpha menyimpan batas terbaik untuk MAX, beta menyimpan batas terbaik untuk MIN. Begitu kedua batas ini saling bertemu, eksplorasi cabang bisa dihentikan.*

**Efisiensi:**
- Kasus terbaik: mengurangi node dari O(b^m) menjadi **O(b^(m/2))**
- Pada praktiknya, penghematan sangat dipengaruhi oleh urutan child yang dievaluasi
- Hasil keputusan tetap sama dengan Minimax; yang berubah hanya jumlah node yang dikunjungi

### 4.3 Contoh Manual Alpha-Beta Pruning

Perhatikan game tree yang sama seperti pada contoh Minimax:

```
                    A (MAX)   α=-∞, β=+∞
                   /   |   \
              B(MIN) C(MIN) D(MIN)
              /  \   /  \   /  \
             3   5  2   9  1    7
```

**Penelusuran langkah demi langkah:**

1. Evaluasi node **B (MIN)**: leaf 3 dan 5 menghasilkan min(3, 5) = **3**. Root menyimpan α = 3.
2. Evaluasi node **C (MIN)**: leaf pertama bernilai 2. Karena 2 ≤ α(3), sisa child **H = 9** tidak perlu dicek. Node C dipangkas dan bernilai **2**.
3. Evaluasi node **D (MIN)**: leaf pertama bernilai 1. Karena 1 ≤ α(3), sisa child **J = 7** juga dipangkas. Node D bernilai **1**.
4. Root **A (MAX)** memilih max(3, 2, 1) = **3**.

Jadi, Alpha-Beta Pruning menghasilkan nilai akhir yang sama dengan Minimax, tetapi dengan evaluasi node yang lebih sedikit karena beberapa cabang dihentikan lebih awal.

### 4.4 Implementasi Alpha-Beta Pruning

Implementasi Alpha-Beta Pruning pada notebook ini mengikuti bentuk generik Minimax: parameter `alpha` dan `beta` dibawa turun ke recursion, lalu tiap node berhenti dieksplorasi ketika batasnya sudah terpenuhi.

In [7]:
def alphabeta(node, depth, alpha, beta, maximizingPlayer):

    # Base case
    if node not in tree or depth == 0:
        return node

    if maximizingPlayer:
        bestValue = float('-inf')
        for child in tree[node]:
            value = alphabeta(child, depth - 1, alpha, beta, False)
            bestValue = max(bestValue, value)
            alpha = max(alpha, bestValue)
            # Pruning
            if beta <= alpha:
                break
        return bestValue

    else:
        bestValue = float('inf')
        for child in tree[node]:
            value = alphabeta(child, depth - 1, alpha, beta, True)
            bestValue = min(bestValue, value)
            beta = min(beta, bestValue)
            # Pruning
            if beta <= alpha:
                break
        return bestValue

In [8]:
def find_best_move_ab(node, depth):

    bestValue = float('-inf')
    bestMove = None

    alpha = float('-inf')
    beta = float('inf')

    for child in tree[node]:
        value = alphabeta(child, depth - 1, alpha, beta, False)

        if value > bestValue:
            bestValue = value
            bestMove = child

        alpha = max(alpha, bestValue)

    return bestMove, bestValue

In [9]:
print("Minimax:", minimax('A', 3, True))

print("Alpha-Beta:", alphabeta('A', 3, float('-inf'), float('inf'), True))

Minimax: 6
Alpha-Beta: 6


### 4.5 Visualisasi Proses Minimax

In [10]:
def alphabeta_trace(node, depth, alpha, beta, maximizingPlayer, indent=""):

    if node not in tree or depth == 0:
        print(f"{indent}{node}")
        return node
    print(f"{indent}{node} ({'MAX' if maximizingPlayer else 'MIN'}) [α={alpha}, β={beta}]")

    if maximizingPlayer:
        bestValue = float('-inf')
        for child in tree[node]:
            value = alphabeta_trace(child, depth - 1, alpha, beta, False, indent + "  ")
            bestValue = max(bestValue, value)
            alpha = max(alpha, bestValue)
            if beta <= alpha:
                print(f"{indent} PRUNE (β ≤ α)")
                break
        print(f"{indent}-> pilih {bestValue}")
        return bestValue

    else:
        bestValue = float('inf')
        for child in tree[node]:
            value = alphabeta_trace(child, depth - 1, alpha, beta, True, indent + "  ")
            bestValue = min(bestValue, value)
            beta = min(beta, bestValue)
            if beta <= alpha:
                print(f"{indent} PRUNE (β ≤ α)")
                break
        print(f"{indent}-> pilih {bestValue}")
        return bestValue

In [11]:
result = alphabeta_trace('A', 3, float('-inf'), float('inf'), True)
print("\nNilai optimal:", result)

A (MAX) [α=-inf, β=inf]
  B (MIN) [α=-inf, β=inf]
    E (MAX) [α=-inf, β=inf]
      8
      5
    -> pilih 8
    F (MAX) [α=-inf, β=8]
      6
      -4
    -> pilih 6
  -> pilih 6
  C (MIN) [α=6, β=inf]
    G (MAX) [α=6, β=inf]
      3
      8
    -> pilih 8
    H (MAX) [α=6, β=8]
      4
      -6
    -> pilih 4
   PRUNE (β ≤ α)
  -> pilih 4
  D (MIN) [α=6, β=inf]
    I (MAX) [α=6, β=inf]
      1
      2
    -> pilih 2
   PRUNE (β ≤ α)
  -> pilih 2
-> pilih 6

Nilai optimal: 6


### 4.5 Penghitung Node (Minimax & Alpha Beta Pruning)

Bagian berikut menambahkan **penghitung node** agar kita bisa melihat perbedaan jumlah evaluasi antara Minimax biasa dan Alpha-Beta Pruning pada tree yang sama. Hasil akhirnya tetap identik, tetapi Alpha-Beta biasanya mengunjungi lebih sedikit node.

In [12]:
# Versi Minimax dengan penghitung node untuk perbandingan
node_count_minimax = 0

def minimax_counted(tree, node, is_max):
    global node_count_minimax
    node_count_minimax += 1

    if node not in tree or isinstance(tree[node], int):
        return node if node not in tree else tree[node]
    children = tree[node]

    if is_max:
        best_score = float('-inf')
        for child in children:
            score = minimax_counted(tree, child, False)
            best_score = max(best_score, score)
        return best_score
    else:
        best_score = float('inf')
        for child in children:
            score = minimax_counted(tree, child, True)
            best_score = min(best_score, score)
        return best_score

In [13]:
# Versi Alpha-Beta dengan penghitung node untuk perbandingan
node_count_ab = 0

def alpha_beta_counted(tree, node, is_max, alpha, beta):
    global node_count_ab
    node_count_ab += 1

    if node not in tree or isinstance(tree[node], int):
        return node if node not in tree else tree[node]
    children = tree[node]

    if is_max:
        value = float('-inf')
        for child in children:
            value = max(value, alpha_beta_counted(tree, child, False, alpha, beta))
            alpha = max(alpha, value)
            if beta <= alpha:
                break
        return value
    else:
        value = float('inf')
        for child in children:
            value = min(value, alpha_beta_counted(tree, child, True, alpha, beta))
            beta = min(beta, value)
            if beta <= alpha:
                break
        return value

### 4.6 Perbandingan Minimax vs Alpha-Beta Pruning

Jalankan cell berikut untuk melihat perbedaan jumlah node yang dievaluasi oleh kedua algoritma pada kondisi board yang sama.

In [14]:
# Study case tree yang sama seperti pada Minimax

study_tree = {
    'A': ['B', 'C', 'D'],

    'B': ['E', 'F'],
    'C': ['G', 'H'],
    'D': ['I', 'J'],

    'E': [8, 5],
    'F': [6, -4],
    'G': [3, 8],
    'H': [4, -6],
    'I': [1, 2],
    'J': [5, 2]
}

# Reset counter
node_count_minimax = 0
node_count_ab = 0

# Jalankan Minimax biasa pada game tree yang sama
nilai_minimax = minimax_counted(study_tree, 'A', True)

# Jalankan Alpha-Beta Pruning pada game tree yang sama
nilai_ab = alpha_beta_counted(study_tree, 'A', True, float('-inf'), float('inf'))

print("Perbandingan jumlah node yang dievaluasi (game_tree):")
print(f"  Minimax biasa       : {node_count_minimax} node")
print(f"  Alpha-Beta Pruning  : {node_count_ab} node")
print(f"  Penghematan         : {node_count_minimax - node_count_ab} node ({((node_count_minimax - node_count_ab)/node_count_minimax * 100):.1f}%)")
print(f"  Nilai Minimax       : {nilai_minimax}")
print(f"  Nilai Alpha-Beta    : {nilai_ab}")
print("  Hasil sama          :", nilai_minimax == nilai_ab)

Perbandingan jumlah node yang dievaluasi (game_tree):
  Minimax biasa       : 22 node
  Alpha-Beta Pruning  : 19 node
  Penghematan         : 3 node (13.6%)
  Nilai Minimax       : 6
  Nilai Alpha-Beta    : 6
  Hasil sama          : True


In [15]:
# Tree latihan yang sama seperti pada bagian latihan soal
game_tree_latihan = {
    'Root': ['A', 'B'],
    'A': ['A1', 'A2'],
    'B': ['B1', 'B2'],
    'A1': ['A1a', 'A1b'],
    'A2': ['A2a', 'A2b'],
    'B1': ['B1a', 'B1b'],
    'B2': ['B2a', 'B2b'],
    'A1a': 4,
    'A1b': 6,
    'A2a': 2,
    'A2b': 8,
    'B1a': 5,
    'B1b': 3,
    'B2a': 7,
    'B2b': 1
}

# Contoh tambahan pada tree latihan untuk memverifikasi hasil yang sama
node_count_minimax = 0
node_count_ab = 0

nilai_minimax_latihan = minimax_counted(game_tree_latihan, 'Root', True)
nilai_ab_latihan = alpha_beta_counted(game_tree_latihan, 'Root', True, float('-inf'), float('inf'))

print("Perbandingan pada game_tree_latihan:")
print(f"  Minimax biasa       : {node_count_minimax} node")
print(f"  Alpha-Beta Pruning  : {node_count_ab} node")
print(f"  Nilai Minimax       : {nilai_minimax_latihan}")
print(f"  Nilai Alpha-Beta    : {nilai_ab_latihan}")
print("  Hasil sama          :", nilai_minimax_latihan == nilai_ab_latihan)

Perbandingan pada game_tree_latihan:
  Minimax biasa       : 15 node
  Alpha-Beta Pruning  : 12 node
  Nilai Minimax       : 6
  Nilai Alpha-Beta    : 6
  Hasil sama          : True


## 5. Ringkasan dan Perbandingan

### 5.1 Tabel Perbandingan Minimax vs Alpha-Beta Pruning

| Aspek | Minimax | Alpha-Beta Pruning |
|---|---|---|
| **Ide Dasar** | Eksplorasi seluruh game tree | Minimax + pemangkasan cabang tidak relevan |
| **Hasil Keputusan** | Optimal | Optimal (identik dengan Minimax) |
| **Node Dievaluasi** | O(b^m) | O(b^(m/2)) terbaik |
| **Parameter Tambahan** | Tidak ada | Alpha (α) dan Beta (β) |
| **Kondisi Pruning** | Tidak ada | β ≤ α → stop eksplorasi |
| **Cocok untuk** | Pembelajaran konsep | Implementasi nyata (game AI) |

### 5.2 Kapan Menggunakan Adversarial Search?

Gunakan Adversarial Search ketika:
- Ada **dua atau lebih agen** yang bersaing
- Lingkungan bersifat **deterministik** dan **fully observable**
- Setiap agen bermain **bergantian** (turn-based)
- Tujuan setiap agen **berlawanan** (zero-sum)

Contoh aplikasi dunia nyata:
- **Game AI**: catur, checkers, Go, tic-tac-toe
- **Negosiasi otomatis**: sistem yang mewakili pihak tertentu dalam negosiasi
- **Cybersecurity**: simulasi penyerang vs pertahanan
- **Trading otomatis**: simulasi kompetisi pasar

## 6. Latihan Soal
<font color="blue">Latihan 4</font>

### PR — Tic-Tac-Toe dengan Minimax & Alpha-Beta
Tujuan
- Mengimplementasikan Minimax
- Mengimplementasikan Alpha-Beta Pruning
- Menentukan langkah terbaik pada Tic-Tac-Toe

Instruksi
- Lengkapi bagian kode yang kosong (_____)
- Gunakan:
   - MAX = 'X'
   - MIN = 'O'

### 1. Setup

In [7]:
EMPTY = ' '
MAX = 'X'
MIN = 'O'

### 2. Helper Functions

In [2]:
import copy

def get_moves(board):
    return [(r, c) for r in range(3) for c in range(3) if board[r][c] == EMPTY]

def apply_move(board, move, player):
    new_board = copy.deepcopy(board)
    r, c = move
    new_board[r][c] = player
    return new_board

def check_winner(board):
    lines = []
    lines.extend(board)

    for c in range(3):
        lines.append([board[r][c] for r in range(3)])

    lines.append([board[i][i] for i in range(3)])
    lines.append([board[i][2-i] for i in range(3)])

    for line in lines:
        if line[0] != EMPTY and line.count(line[0]) == 3:
            return line[0]
    return None

def is_full(board):
    return all(cell != EMPTY for row in board for cell in row)

def evaluate(winner):
    if winner == MAX:
        return 1
    elif winner == MIN:
        return -1
    return 0

### 3. Minimax

In [8]:
def minimax(board, is_maximizing):
    winner = check_winner(board)

    # TERMINAL
    if winner is not None:
        return evaluate(winner)

    if is_full(board):
        return 0

    if is_maximizing:
        bestValue = float('-inf')
        for move in get_moves(board):
            value = minimax(apply_move(board, move, MAX), False)
            bestValue = max(bestValue, value)
        return bestValue

    else:
        bestValue = float('inf')
        for move in get_moves(board):
            value = minimax(apply_move(board, move, MIN), True)
            bestValue = min(bestValue, value)
        return bestValue

### 4. Best Move

In [9]:
def find_best_move(board):
    bestValue = float('-inf')
    bestMove = None
    for move in get_moves(board):
        value = minimax(apply_move(board, move, MAX), False)
        if value > bestValue:
            bestValue = value
            bestMove = move
    return bestMove

### 5. Alpha-Beta Pruning

In [10]:
def alphabeta(board, alpha, beta, is_maximizing):
    winner = check_winner(board)
    if winner is not None:
        return evaluate(winner)
    if is_full(board):
        return 0
    if is_maximizing:
        bestValue = float('-inf')
        for move in get_moves(board):
            value = alphabeta(apply_move(board, move, MAX), alpha, beta, False)
            bestValue = max(bestValue, value)
            alpha = max(alpha, bestValue)
            if beta <= alpha:
                break
        return bestValue
    else:
        bestValue = float('inf')
        for move in get_moves(board):
            value = alphabeta(apply_move(board, move, MIN), alpha, beta, True)
            bestValue = min(bestValue, value)
            beta = min(beta, bestValue)
            if beta <= alpha:
                break
        return bestValue

### 6. Uji Coba

In [11]:
board = [
    ['X', 'O', 'X'],
    [' ', 'O', ' '],
    [' ', ' ', '']
]

print(find_best_move(board))

(2, 1)
